# RSNA 2026 Knee Abnormality Detection — Multi-Model Ensemble Submission

> 集成 5 个独立训练的模型生成最终提交文件。在 Kaggle 环境中运行，需要挂载以下 inputs:
>
> | Input | 来源 | 内容 |
> |-------|------|------|
> | Competition Dataset | RSNA 2026 Knee | `train_series/`, `test_series/`, `train.csv`, `sample_submission.csv` |
> | Source Code Bundle | Kaggle Dataset | 本项目完整源码 (models, datasets, losses, utils, configs) |
> | Trained Checkpoints | Kaggle Dataset | 各模型的最佳 checkpoint (`.pt` / `.pth`) |

## 模型阵容 & 权重

| # | 模型 | Backbone | 输入 | 权重 | Val AUC |
|---|------|----------|------|------|------|
| 1 | `EfficientNetV2S25D` | EfficientNetV2-S | 5-slice Sagittal 2.5D | 0.25 | — |
| 2 | `ConvNeXt25D` | ConvNeXt-S | 5-slice Sagittal 2.5D | 0.15 | — |
| 3 | `Swin25D` | Swin-T | 5-slice Sagittal 2.5D | 0.10 | — |
| 4 | `TriPlaneModel` | Shared EffNetV2-S | Sag+Cor+Ax 2.5D | 0.30 | — |
| 5 | `ResNet3DModel` | ResNet3D-18 | 32-slice 3D volume | 0.20 | — |

## 推理策略

- **2.5D 模型**: 逐 slice 推理 → top-25% mean 聚合至 study 级
- **3D 模型**: 整个 volume 输入 → 直接输出 study 级 logits
- **Tri-plane 模型**: 三平面同时输入 → 逐 slice → study 聚合
- **集成**: 加权平均各模型的 sigmoid 概率

## 关键约束

- Kaggle GPU: T4 × 2 (16GB VRAM each) 或 P100 (16GB)
- 测试集: ~200 studies, ~1,000 series → 推理时间 < 30 min
- 输出: `submission.csv` (StudyInstanceUID × 12 概率)

In [ ]:
# ============================================================
# Cell 1: 环境 & 导入
# ============================================================
from __future__ import annotations
import gc, sys, time, warnings
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

# ── 设备 ────────────────────────────────────────────────────
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"
print(f"Device: {DEVICE}")
if DEVICE.type == "cuda":
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        print(f"  GPU {i}: {props.name} ({props.total_mem / 1024**3:.1f} GB)")

# ── 常量 ────────────────────────────────────────────────────
TARGETS = [
    "ACL", "MCL",
    "Medial Meniscus", "Lateral Meniscus",
    "Medial OA", "Lateral OA", "PF OA",
    "Effusion", "Synovitis", "Baker's",
    "Contusion", "Fracture",
]
IMAGE_SIZE = 384
NUM_SLICES = 5
VOLUME_DEPTH = 32
VOLUME_SIZE = 128
BATCH_SIZE_2D = 8         # 2.5D 模型推理 batch size
BATCH_SIZE_3D = 1         # 3D 模型推理 batch size (显存约束)
TOPK_FRACTION = 0.25      # slice → study 聚合比例

print(f"IMG_SIZE={IMAGE_SIZE}  SLICES={NUM_SLICES}  TOP_K={TOPK_FRACTION:.0%}")
print(f"BATCH_2D={BATCH_SIZE_2D}  BATCH_3D={BATCH_SIZE_3D}")

In [ ]:
# ============================================================
# Cell 2: 文件发现 — 自动定位 Kaggle input
# ============================================================
INPUT = Path("/kaggle/input")

def find_one(pattern: str, required: bool = True) -> Path | None:
    """在 INPUT 目录下递归查找文件/目录."""
    hits = list(INPUT.rglob(pattern))
    if required and not hits:
        raise FileNotFoundError(
            f"找不到 '{pattern}' — 请在 Kaggle notebook 右侧挂载对应 Dataset"
        )
    return hits[0] if hits else None

# ── Competition data ────────────────────────────────────────
sample_csv = find_one("sample_submission.csv")
test_series_csv = find_one("test_series.csv")

# test_series DICOM 根目录
test_series_root = next(
    (p for p in INPUT.rglob("test_series") if p.is_dir()), None
)
if test_series_root is None:
    raise FileNotFoundError("找不到 test_series DICOM 目录")

# ── Source code bundle ──────────────────────────────────────
source_root = next(
    (p.parent for p in INPUT.rglob("utils.py") if p.parent.name != "__pycache__"),
    None,
)
if source_root is None:
    raise FileNotFoundError(
        "找不到源码 bundle — 请将项目文件打包成 Kaggle Dataset 并挂载"
    )
if str(source_root) not in sys.path:
    sys.path.insert(0, str(source_root))

# ── Trained checkpoints ─────────────────────────────────────
eff_ckpt   = find_one("efficientnetv2_s_best.pt", required=False)
conv_ckpt  = find_one("convnext_small_best.pt", required=False)
swin_ckpt  = find_one("swin_tiny_best.pt", required=False)
tri_ckpt   = find_one("triplane_best.pt", required=False)
res3d_ckpt = find_one("resnet3d_best.pt", required=False)

print("=== 文件发现结果 ===")
print(f"  sample_submission:  {sample_csv}")
print(f"  test_series.csv:    {test_series_csv}")
print(f"  test_series DICOM:  {test_series_root}")
print(f"  source code:        {source_root}")
print(f"  EfficientNetV2-S:   {eff_ckpt.name if eff_ckpt else 'NOT FOUND'}")
print(f"  ConvNeXt-S:         {conv_ckpt.name if conv_ckpt else 'NOT FOUND'}")
print(f"  Swin-T:             {swin_ckpt.name if swin_ckpt else 'NOT FOUND'}")
print(f"  TriPlane:           {tri_ckpt.name if tri_ckpt else 'NOT FOUND'}")
print(f"  ResNet3D-18:        {res3d_ckpt.name if res3d_ckpt else 'NOT FOUND'}")

# 至少需要一个 checkpoint
available = [c for c in [eff_ckpt, conv_ckpt, swin_ckpt, tri_ckpt, res3d_ckpt] if c is not None]
if not available:
    raise FileNotFoundError("没有任何 checkpoint！请至少挂载一个训练好的模型权重")
print(f"\n可用模型: {len(available)} / 5")

In [ ]:
# ============================================================
# Cell 3: 导入项目模块
# ============================================================
# 从源码 bundle 导入所有需要的模块

from datasets.dataset import Knee25DDataset
from datasets.triplane_dataset import TriPlaneDataset
from datasets.volume_dataset import VolumeDataset
from datasets.dicom_loader import read_dicom_series

from models.efficientnet25d import EfficientNetV2S25D
from models.convnext import ConvNeXt25D
from models.swin import Swin25D
from models.triplane import TriPlaneModel
from models.resnet3d import ResNet3DModel
from models.head import ClassificationHead

from utils import (
    TARGET_COLUMNS,
    aggregate_to_study,
    compute_macro_auc,
    compute_per_class_auc,
    format_per_class_auc,
)

print("所有模块导入成功 ✓")

In [ ]:
# ============================================================
# Cell 4: 测试集元数据分析
# ============================================================
test_series_df = pd.read_csv(test_series_csv)

# 验证必要列
required_cols = {"StudyInstanceUID", "SeriesInstanceUID", "Anatomical_Plane"}
if missing_cols := required_cols - set(test_series_df.columns):
    raise ValueError(f"test_series.csv 缺少列: {missing_cols}")

test_study_uids = sorted(test_series_df["StudyInstanceUID"].unique())
n_studies = len(test_study_uids)
n_series = len(test_series_df)

print(f"=== 测试集概览 ===")
print(f"  Studies:        {n_studies:,}")
print(f"  Series:         {n_series:,}")
print(f"  Series/Study:   {n_series / max(n_studies, 1):.1f}")

# 平面分布
plane_counts = test_series_df["Anatomical_Plane"].value_counts()
print(f"\n  平面分布:")
for plane in ["Sagittal", "Coronal", "Axial"]:
    print(f"    {plane:<12s} {plane_counts.get(plane, 0):,} series")

# 三平面覆盖率
study_planes = test_series_df.groupby("StudyInstanceUID")["Anatomical_Plane"].apply(set)
all_three = study_planes.apply(lambda s: {"Sagittal", "Coronal", "Axial"}.issubset(s))
print(f"\n  三平面齐全: {all_three.sum():,} / {n_studies:,} ({all_three.mean()*100:.1f}%)")

# 缺失情况
for plane in ["Coronal", "Axial"]:
    missing = study_planes.apply(lambda s, p=plane: p not in s)
    print(f"  缺 {plane:<12s} {missing.sum():,} studies ({missing.mean()*100:.1f}%)")

In [ ]:
# ============================================================
# Cell 5: 构建测试 Datasets
# ============================================================
# 为每个模型类型构建对应的 dataset
#   - 2.5D 模型: Knee25DDataset (Sagittal only)
#   - Tri-plane: TriPlaneDataset (Sag + Cor + Ax)
#   - 3D: VolumeDataset (Sagittal volume)

# 构建 dummy labels (推理时全填 0, 仅用于 dataset 索引)
dummy_labels = pd.DataFrame({
    "StudyInstanceUID": test_study_uids,
    **{col: 0 for col in TARGETS},
}).set_index("StudyInstanceUID")

# ── 2.5D Sagittal Dataset (EffNet/ConvNeXt/Swin 共用) ──────
dataset_2d = None
if any([eff_ckpt, conv_ckpt, swin_ckpt]):
    dataset_2d = Knee25DDataset(
        series_df=test_series_df,
        labels_df=dummy_labels,
        dicom_root=str(test_series_root),
        planes=["Sagittal"],
        image_size=IMAGE_SIZE,
        slice_count=NUM_SLICES,
        is_train=False,
    )
    print(f"2.5D Sagittal Dataset: {len(dataset_2d):,} slices")
else:
    print("2.5D Dataset: SKIP (无 checkpoint)")

# ── Tri-Plane Dataset (三平面) ─────────────────────────────
dataset_tri = None
if tri_ckpt is not None:
    dataset_tri = TriPlaneDataset(
        series_df=test_series_df,
        labels_df=dummy_labels,
        dicom_root=str(test_series_root),
        image_size=IMAGE_SIZE,
        slice_count=NUM_SLICES,
        planes=["Sagittal", "Coronal", "Axial"],
        is_train=False,
    )
    print(f"TriPlane Dataset:        {len(dataset_tri):,} slices")
else:
    print("TriPlane Dataset:        SKIP (无 checkpoint)")

# ── 3D Volume Dataset ──────────────────────────────────────
dataset_3d = None
if res3d_ckpt is not None:
    dataset_3d = VolumeDataset(
        series_df=test_series_df,
        labels_df=dummy_labels,
        dicom_root=str(test_series_root),
        volume_depth=VOLUME_DEPTH,
        volume_size=VOLUME_SIZE,
        plane="Sagittal",
        is_train=False,
    )
    print(f"3D Volume Dataset:       {len(dataset_3d):,} volumes")
else:
    print("3D Volume Dataset:       SKIP (无 checkpoint)")

print(f"\n✓ 所有 dataset 构建完成")

In [ ]:
# ============================================================
# Cell 6: 构建 DataLoaders
# ============================================================

loaders = {}

if dataset_2d is not None:
    loaders["2d"] = DataLoader(
        dataset_2d,
        batch_size=BATCH_SIZE_2D,
        shuffle=False,
        num_workers=2,
        pin_memory=True,
    )
    print(f"Loader 2D:       {len(loaders['2d'])} batches (bs={BATCH_SIZE_2D})")

if dataset_tri is not None:
    loaders["tri"] = DataLoader(
        dataset_tri,
        batch_size=BATCH_SIZE_2D,
        shuffle=False,
        num_workers=2,
        pin_memory=True,
    )
    print(f"Loader TriPlane: {len(loaders['tri'])} batches (bs={BATCH_SIZE_2D})")

if dataset_3d is not None:
    loaders["3d"] = DataLoader(
        dataset_3d,
        batch_size=BATCH_SIZE_3D,
        shuffle=False,
        num_workers=1,
        pin_memory=True,
    )
    print(f"Loader 3D:       {len(loaders['3d'])} batches (bs={BATCH_SIZE_3D})")

print(f"\n✓ {len(loaders)} DataLoader(s) 就绪")

In [ ]:
# ============================================================
# Cell 7: 加载训练好的模型
# ============================================================

def load_checkpoint(model: nn.Module, ckpt_path: Path, name: str) -> nn.Module | None:
    """加载 checkpoint, 处理多种保存格式."""
    ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)

    # 尝试多种 key 格式
    state = ckpt.get("model", ckpt.get("model_state_dict", ckpt.get("state_dict", ckpt)))

    # 去掉 DataParallel 的 'module.' 前缀
    state = {k.replace("module.", ""): v for k, v in state.items()}

    missing, unexpected = model.load_state_dict(state, strict=False)
    if unexpected:
        print(f"  ⚠️  {name}: {len(unexpected)} unexpected keys (ignored)")
    if missing:
        print(f"  ⚠️  {name}: {len(missing)} missing keys")

    val_auc = ckpt.get("auc", ckpt.get("val_auc", 0))
    epoch = ckpt.get("epoch", "?")
    print(f"  ✓ {name}: epoch={epoch}, val_auc={val_auc:.4f}")

    return model.to(DEVICE).eval()


models = {}  # name → (model, weight)

# ── 1. EfficientNetV2-S 2.5D ────────────────────────────────────
if eff_ckpt is not None:
    eff_model = EfficientNetV2S25D(
        in_channels=5, num_classes=12, pretrained=False, dropout=0.3,
    )
    models["effnet"] = (load_checkpoint(eff_model, eff_ckpt, "EfficientNetV2-S"), 0.25)

# ── 2. ConvNeXt-S 2.5D ──────────────────────────────────────────
if conv_ckpt is not None:
    conv_model = ConvNeXt25D(
        in_channels=5, num_classes=12, pretrained=False, dropout=0.3,
    )
    models["convnext"] = (load_checkpoint(conv_model, conv_ckpt, "ConvNeXt-S"), 0.15)

# ── 3. Swin-T 2.5D ──────────────────────────────────────────────
if swin_ckpt is not None:
    swin_model = Swin25D(
        in_channels=5, num_classes=12, pretrained=False, dropout=0.3,
    )
    models["swin"] = (load_checkpoint(swin_model, swin_ckpt, "Swin-T"), 0.10)

# ── 4. TriPlane Model ───────────────────────────────────────────
if tri_ckpt is not None:
    tri_model = TriPlaneModel(
        in_channels=5, num_classes=12, pretrained=False,
        dropout=0.3, shared_backbone=True, fusion="concat",
    )
    models["triplane"] = (load_checkpoint(tri_model, tri_ckpt, "TriPlane"), 0.30)

# ── 5. ResNet3D-18 ──────────────────────────────────────────────
if res3d_ckpt is not None:
    res3d_model = ResNet3DModel(
        in_channels=1, num_classes=12, pretrained=False,
        dropout=0.3, use_grad_checkpoint=False,  # 推理不需要 grad ckpt
    )
    models["resnet3d"] = (load_checkpoint(res3d_model, res3d_ckpt, "ResNet3D-18"), 0.20)

# ── 归一化权重 ──────────────────────────────────────────────────
total_w = sum(w for _, w in models.values())
for name in models:
    m, w = models[name]
    models[name] = (m, w / total_w)

print(f"\n=== 集成权重 (归一化后) ===")
for name, (m, w) in models.items():
    n_params = sum(p.numel() for p in m.parameters()) / 1e6
    print(f"  {name:<12s} weight={w:.3f}  params={n_params:.1f}M")
print(f"\n✓ {len(models)} 模型加载完成")

In [ ]:
# ============================================================
# Cell 8: 推理 — 2.5D 单平面模型 (EffNet / ConvNeXt / Swin)
# ============================================================
# 策略: 逐 slice 推理 → 收集每个 study 的所有 slice logits
#        推理结束后用 top-K mean 聚合至 study 级

# 收集 slice 级 logits (所有 2.5D 模型共用同一个 loader)
slice_logits_2d: dict[str, dict[str, list[np.ndarray]]] = defaultdict(
    lambda: defaultdict(list)
)  # model_name → study_uid → [logits_from_each_batch]

if "2d" in loaders:
    loader_2d = loaders["2d"]
    n_batches = len(loader_2d)
    print(f"2.5D 推理: {n_batches} batches, {len(dataset_2d):,} slices")

    # 只取需要 2.5D 数据的前 3 个模型
    models_2d = {k: v for k, v in models.items() if k in ("effnet", "convnext", "swin")}
    print(f"  参与模型: {list(models_2d.keys())}")
    print(f"{'='*55}")

    with torch.inference_mode():
        for batch in tqdm(loader_2d, desc="2.5D inference", unit="batch"):
            images = batch["image"].to(DEVICE, non_blocking=True)
            study_uids = batch["study_uid"]

            with torch.autocast(
                device_type="cuda", dtype=torch.float16, enabled=USE_AMP
            ):
                for mname, (model, _) in models_2d.items():
                    logits = model(images).float().cpu().numpy()  # [B, 12]
                    for j, uid in enumerate(study_uids):
                        slice_logits_2d[mname][str(uid)].append(logits[j])

    print(f"\n✓ 2.5D 推理完成，收集到的 study 数:")
    for mname in models_2d:
        n_s = len(slice_logits_2d[mname])
        n_total = sum(len(v) for v in slice_logits_2d[mname].values())
        print(f"    {mname:<12s} {n_s} studies, {n_total:,} slices")
else:
    print("⏭  跳过 2.5D 推理 — 无对应模型")

In [ ]:
# ============================================================
# Cell 9: 推理 — Tri-Plane 模型
# ============================================================
# 三平面同时推理: 每个 batch 返回 sag/cor/ax 三个 tensor

slice_logits_tri: dict[str, list[np.ndarray]] = defaultdict(list)
  # study_uid → [logits_from_each_slice]

if "tri" in loaders and "triplane" in models:
    loader_tri = loaders["tri"]
    tri_model, tri_weight = models["triplane"]
    print(f"Tri-Plane 推理: {len(loader_tri)} batches, {len(dataset_tri):,} slices")
    print(f"{'='*55}")

    with torch.inference_mode():
        for batch in tqdm(loader_tri, desc="TriPlane inference", unit="batch"):
            sag = batch["sag"].to(DEVICE, non_blocking=True)
            cor = batch["cor"].to(DEVICE, non_blocking=True)
            ax = batch["ax"].to(DEVICE, non_blocking=True)
            study_uids = batch["study_uid"]

            with torch.autocast(
                device_type="cuda", dtype=torch.float16, enabled=USE_AMP
            ):
                logits = tri_model(sag, cor, ax).float().cpu().numpy()  # [B, 12]

            for j, uid in enumerate(study_uids):
                slice_logits_tri[str(uid)].append(logits[j])

    n_studies_tri = len(slice_logits_tri)
    n_slices_tri = sum(len(v) for v in slice_logits_tri.values())
    print(f"\n✓ TriPlane 推理完成: {n_studies_tri} studies, {n_slices_tri:,} slices")
else:
    print("⏭  跳过 TriPlane 推理 — 无 checkpoint 或无 loader")

In [ ]:
# ============================================================
# Cell 10: 推理 — 3D ResNet 模型
# ============================================================
# 3D volume 一次推理直接输出 study 级 logits (无需 slice 聚合)

study_logits_3d: dict[str, np.ndarray] = {}  # study_uid → logits [12]

if "3d" in loaders and "resnet3d" in models:
    loader_3d = loaders["3d"]
    res3d_model, res3d_weight = models["resnet3d"]
    print(f"3D 推理: {len(loader_3d)} volumes, bs={BATCH_SIZE_3D}")
    print(f"{'='*55}")

    with torch.inference_mode():
        for batch in tqdm(loader_3d, desc="3D inference", unit="volume"):
            volume = batch["volume"].to(DEVICE, non_blocking=True)  # [B, 1, D, H, W]
            study_uid = batch["study_uid"]

            with torch.autocast(
                device_type="cuda", dtype=torch.float16, enabled=USE_AMP
            ):
                logits = res3d_model(volume).float().cpu().numpy()  # [B, 12]

            for j, uid in enumerate(study_uid):
                study_logits_3d[str(uid)] = logits[j]

    print(f"\n✓ 3D 推理完成: {len(study_logits_3d)} studies")
else:
    print("⏭  跳过 3D 推理 — 无 checkpoint 或无 loader")

In [ ]:
# ============================================================
# Cell 11: Slice → Study 聚合 (2.5D & TriPlane)
# ============================================================
# 将逐 slice logits 聚合为 study 级 logits (top-K mean)

# study_uid → {model_name → aggregated_logits [12]}
study_logits_all: dict[str, dict[str, np.ndarray]] = defaultdict(dict)

# ── 聚合 2.5D 模型的 slice logits ──────────────────────────
for mname, mdict in slice_logits_2d.items():
    for uid, logit_list in mdict.items():
        stacked = np.stack(logit_list)  # [K, 12]
        k = max(1, int(len(stacked) * TOPK_FRACTION))
        top_vals = np.sort(stacked, axis=0)[-k:]
        study_logits_all[uid][mname] = top_vals.mean(axis=0)
    print(f"  {mname:<12s} slice → study: {len(mdict):,} studies aggregated")

# ── 聚合 TriPlane 模型的 slice logits ───────────────────────
for uid, logit_list in slice_logits_tri.items():
    stacked = np.stack(logit_list)  # [K, 12]
    k = max(1, int(len(stacked) * TOPK_FRACTION))
    top_vals = np.sort(stacked, axis=0)[-k:]
    study_logits_all[uid]["triplane"] = top_vals.mean(axis=0)

if slice_logits_tri:
    print(f"  triplane      slice → study: {len(slice_logits_tri):,} studies aggregated")

# ── 加入 3D 模型的 study 级 logits ──────────────────────────
for uid, logits in study_logits_3d.items():
    study_logits_all[uid]["resnet3d"] = logits

if study_logits_3d:
    print(f"  resnet3d      already study-level: {len(study_logits_3d):,} studies")

# ── 统计覆盖率 ─────────────────────────────────────────────
n_aggregated = len(study_logits_all)
n_expected = len(test_study_uids)
print(f"\n聚合完成: {n_aggregated:,} / {n_expected:,} studies 有推理结果")
if n_aggregated < n_expected:
    missing_set = set(test_study_uids) - set(study_logits_all.keys())
    print(f"  ⚠️  {len(missing_set)} studies 无任何模型结果 (将填充 0.5)")

In [ ]:
# ============================================================
# Cell 12: 加权平均集成 — 合并各模型 study 级 logits
# ============================================================
# 对每个 study:
#   1. 收集各模型对其的 logits (可能部分模型缺失)
#   2. 按模型权重加权平均
#   3. Sigmoid → 概率

model_weights = {name: w for name, (_, w) in models.items()}

ensemble_logits = {}   # study_uid → [12] weighted logits
ensemble_contrib = {}  # study_uid → list of contributing model names

for uid in test_study_uids:
    model_logits_for_uid = study_logits_all.get(uid, {})

    if not model_logits_for_uid:
        # 无任何模型覆盖 → 填充 logit=0 (概率 0.5)
        ensemble_logits[uid] = np.zeros(12, dtype=np.float32)
        ensemble_contrib[uid] = ["NONE"]
        continue

    weighted_sum = np.zeros(12, dtype=np.float32)
    weight_total = 0.0
    contribs = []

    for mname, logits in model_logits_for_uid.items():
        w = model_weights.get(mname, 0.0)
        weighted_sum += logits * w
        weight_total += w
        contribs.append(mname)

    # 归一化 (只对参与模型重新分配权重)
    if weight_total > 0:
        weighted_sum /= weight_total

    ensemble_logits[uid] = weighted_sum
    ensemble_contrib[uid] = contribs

# ── Sigmoid → 概率 ──────────────────────────────────────────
ensemble_probs = {
    uid: 1.0 / (1.0 + np.exp(-np.clip(logits, -30, 30)))
    for uid, logits in ensemble_logits.items()
}

# ── 统计每个 study 的模型覆盖率 ────────────────────────────
contrib_counts = defaultdict(int)
for uid, contribs in ensemble_contrib.items():
    contrib_counts[len(contribs) if "NONE" not in contribs else 0] += 1

print("=== 集成结果统计 ===")
for n_models in sorted(contrib_counts.keys()):
    label = f"{n_models} models" if n_models > 0 else "no model (0.5 fill)"
    print(f"  {label:<20s} {contrib_counts[n_models]:,} studies")

print(f"\n  总 studies: {len(ensemble_probs):,}")

In [ ]:
# ============================================================
# Cell 13: 诊断 — 各模型贡献分析
# ============================================================
# 检查每个模型覆盖了多少 study, 预测分布是否合理

print("=== 各模型 Study 覆盖率 ===")
for mname in models:
    covered = sum(1 for uid in test_study_uids if mname in study_logits_all.get(uid, {}))
    print(f"  {mname:<12s} {covered:5,} / {n_studies:,} ({covered/max(n_studies,1)*100:.1f}%)")

# ── 预测概率分布 ────────────────────────────────────────────
all_probs = np.stack(list(ensemble_probs.values()))  # [N, 12]

print(f"\n=== 集成预测概率分布 ===")
print(f"  Mean:  {all_probs.mean():.3f}")
print(f"  Std:   {all_probs.std():.3f}")
print(f"  Min:   {all_probs.min():.4f}")
print(f"  Max:   {all_probs.max():.4f}")
print(f"  NaN:   {np.isnan(all_probs).sum()}")

# ── 每类正样本率 ────────────────────────────────────────────
print(f"\n=== 每类预测正样本率 (prob > 0.5) ===")
n_total = len(all_probs)
for i, col in enumerate(TARGETS):
    pos_rate = (all_probs[:, i] > 0.5).mean() * 100
    bar = "█" * int(pos_rate / 2)
    print(f"  {col:<20s} {pos_rate:5.1f}%  {bar}")

# ── 概率分布直方图 ──────────────────────────────────────────
import matplotlib.pyplot as plt
fig, axes = plt.subplots(3, 4, figsize=(14, 10))
axes = axes.flatten()
for i, col in enumerate(TARGETS):
    axes[i].hist(all_probs[:, i], bins=30, alpha=0.7, color="steelblue", edgecolor="white")
    axes[i].axvline(0.5, color="red", linestyle="--", alpha=0.5)
    axes[i].set_title(col, fontsize=9)
    axes[i].set_xlim(0, 1)
axes[11].axis("off")
fig.suptitle("Per-Class Probability Distribution (Ensemble)", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Cell 14: 生成 Submission CSV
# ============================================================

# ── 验证 sample submission 格式 ────────────────────────────
sample_sub = pd.read_csv(sample_csv)
expected_cols = ["StudyInstanceUID"] + TARGETS
if list(sample_sub.columns) != expected_cols:
    raise ValueError(
        f"官方 submission 列名变更!\n"
        f"  Expected: {expected_cols}\n"
        f"  Got:      {list(sample_sub.columns)}"
    )

# ── 按样本顺序填充概率 ────────────────────────────────────
output_probs = np.zeros((len(sample_sub), 12), dtype=np.float32)

missing_uids = []
for i, uid in enumerate(sample_sub["StudyInstanceUID"]):
    uid_str = str(uid)
    if uid_str in ensemble_probs:
        output_probs[i] = ensemble_probs[uid_str]
    else:
        # 完全没有推理结果的 study (DICOM 缺失等) → 填充 0.5
        output_probs[i] = 0.5
        missing_uids.append(uid_str)

if missing_uids:
    print(f"⚠️  {len(missing_uids)} studies 完全无推理结果，填充 0.5:")
    for uid in missing_uids[:5]:
        print(f"    {uid}")
    if len(missing_uids) > 5:
        print(f"    ... and {len(missing_uids) - 5} more")

# ── 质量控制 ────────────────────────────────────────────────
assert not np.isnan(output_probs).any(), "❌ Submission 含 NaN!"
assert (output_probs >= 0).all() and (output_probs <= 1).all(), \
    "❌ 概率超出 [0, 1] 范围!"
assert (output_probs.std(axis=0) > 0).all(), \
    "❌ 某列标准差为 0 (所有 study 预测相同概率)!"

print("✓ 质量控制检查全部通过")
print(f"  Shape:  {output_probs.shape}")
print(f"  Range:  [{output_probs.min():.4f}, {output_probs.max():.4f}]")
print(f"  Mean:   {output_probs.mean():.4f}")
print(f"  Std:    {output_probs.std():.4f}")

# ── 保存 ────────────────────────────────────────────────────
submission = sample_sub.copy()
submission[TARGETS] = output_probs
submission.to_csv("/kaggle/working/submission.csv", index=False)

print(f"\n{'='*55}")
print(f"✓ Submission 已保存: /kaggle/working/submission.csv")
print(f"  {len(submission)} studies × 12 classes")
print(f"{'='*55}")

# ── 预览 ────────────────────────────────────────────────────
print(f"\n  前 5 行预览:")
display(submission.head())

In [ ]:
# ============================================================
# Cell 15: 运行时统计 & 清理
# ============================================================
import time as _time

print("=== 运行时统计 ===")
print(f"  Device:       {DEVICE}")
if DEVICE.type == "cuda":
    peak_vram = torch.cuda.max_memory_allocated() / 1024**3
    print(f"  Peak VRAM:    {peak_vram:.1f} GB")

# 模型参数统计
total_params = sum(
    sum(p.numel() for p in m.parameters()) for m, _ in models.values()
) / 1e6
print(f"  Total params: {total_params:.1f}M ({len(models)} models)")
print(f"  Top-K frac:   {TOPK_FRACTION:.0%}")
print(f"  AMP:          {USE_AMP}")

# 清理模型显存
for m, _ in models.values():
    del m
models.clear()
gc.collect()
if DEVICE.type == "cuda":
    torch.cuda.empty_cache()

print("\n✓ 清理完成 — 模型已释放，仅保留 submission CSV")

---
## 使用说明

### Kaggle 环境搭建

1. **创建 Kaggle Dataset (源码 bundle)**:
   ```bash
   # 将项目核心文件打包
   zip -r rsna-knee-source.zip \
       models/ datasets/ losses/ utils.py \
       -x "__pycache__/*" "*.pyc"
   ```
   在 Kaggle 上创建 Dataset 并上传此 zip。

2. **上传训练好的 checkpoint**:
   - 将各模型的最佳 `.pt` 文件上传为独立的 Kaggle Dataset
   - 或打包为一个 Dataset: `rsna-knee-checkpoints/`

3. **创建 Kaggle Notebook**:
   - 挂载 Competiton Dataset (自动)
   - 挂载源码 bundle
   - 挂载 checkpoint Dataset(s)
   - 运行此 notebook

### 本地验证 (无 GPU)

此 notebook 需要 GPU + DICOM 数据才能完整运行。本地可通过以下方式验证代码结构:

```python
# 仅测试导入和数据加载逻辑 (不加载模型权重)
python -c "
import sys; sys.path.insert(0, '.')
from datasets.dataset import Knee25DDataset
from utils import TARGET_COLUMNS
print('Import OK')
"
```

### 调整集成权重

修改 Cell 7 中各模型的权重 (基于 validation AUC):

```python
models["effnet"]   = (..., 0.25)   # 0.649 val AUC
models["triplane"] = (..., 0.30)   # 0.661 val AUC  ← 最高权重
models["resnet3d"] = (..., 0.20)   # 0.642 val AUC
models["convnext"] = (..., 0.15)   # 0.635 val AUC
models["swin"]     = (..., 0.10)   # 0.628 val AUC
```

### 常见问题

| 现象 | 可能原因 | 解决 |
|------|----------|------|
| 找不到源码模块 | bundle 路径不对 | 检查 Cell 2 的 `source_root` 输出 |
| VRAM OOM | batch_size 太大 | 减小 `BATCH_SIZE_2D` (4) 或 `BATCH_SIZE_3D` (1) |
| 某模型推理全 0.5 | checkpoint 加载失败 | 检查 `load_checkpoint` 的 warning 信息 |
| Submission 列不匹配 | 官方格式变更 | Cell 14 会自动检查并报错 |
| 推理时间 > 1h | 模型太多 + T4 显存不足 | 减少模型数或使用 P100 |

### 相关文件

| 文件 | 说明 |
|------|------|
| `train.py` | 完整训练管线 (Phase 1-4) |
| `notebooks/05_phase1_debug.ipynb` | Phase 1 调试 notebook |
| `notebooks/06_phase2_debug.ipynb` | Phase 2 Tri-Plane 调试 |
| `notebooks/07_phase3_4_debug.ipynb` | Phase 3-4 3D+Ensemble 调试 |
| `configs/phase4_ensemble.yaml` | Ensemble 配置文件 |